In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc 
import os 
import anndata as ad
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist
import seaborn as sns
from scipy.spatial.distance import squareform

from scipy.cluster.hierarchy import linkage, to_tree
from scipy.spatial.distance import pdist
from Bio import Phylo
from io import StringIO

In [2]:
scdata = sc.read_h5ad('../sly_data/sly_12_large_modify/multi_modify.h5ad')
scdata

AnnData object with n_obs × n_vars = 21723 × 9911
    obs: 'species', 'tissues', 'samples', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'leiden', 'cluster', 'celltype', 'coarse_celltype', 'eco', 'leiden_R', 'new_leiden', 'new_anno', 'split_label', 'sly_anno'
    var: 'gene_ids', 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'celltype_colors', 'cluster_colors', 'dendrogram_new_anno', 'eco_colors', 'hvg', 'leiden', 'leiden_R', 'leiden_R_colors', 'leiden_colors', 'neighbors', 'new_anno_colors', 'new_leiden_colors', 'pca', 'rank_genes_groups', 'sly_anno_colors', 'species_colors', 'split_label_colors', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    obsp: 'connectivities', 'distances'

In [3]:
def bulk(adata, key, group_col):
    N = adata.obsm[key].shape[1]
    groups = adata.obs[group_col].unique()
    mean_expr = pd.DataFrame(index=[f'embed {i}' for i in range(N)], columns=groups)
    for group in groups:
        subset = adata.obsm[key][adata.obs[group_col] == group]
        mean_expr[group] = subset.mean(0).flatten()
        # # 处理稀疏矩阵或稠密矩阵
        # if isinstance(subset.X, np.ndarray):
        #     mean_expr[group] = subset.X.mean(axis=0)
        # else:
        #     mean_expr[group] = np.array(subset.X.mean(axis=0)).flatten()
    return mean_expr.T


def get_pseudobulk(adata, key, group_col, sample_number = 100):
    N = adata.obsm[key].shape[1]
    groups = adata.obs[group_col].unique()
    mean_expr = pd.DataFrame(index=[f'embed {i}' for i in range(N)], columns=groups)
    for group in groups:
        subset = adata.obsm[key][adata.obs[group_col] == group]
        idx = np.random.randint(0, subset.shape[0], sample_number)
        subset = subset[idx,:]
        mean_expr[group] = subset.mean(0).flatten()
        # # 处理稀疏矩阵或稠密矩阵
        # if isinstance(subset.X, np.ndarray):
        #     mean_expr[group] = subset.X.mean(axis=0)
        # else:
        #     mean_expr[group] = np.array(subset.X.mean(axis=0)).flatten()
    return mean_expr.T

def build_tree_from_data(df, method='complete', metric='correlation'):
    """构建层次聚类树"""
    Z = linkage(df, method=method, metric=metric)
    return Z

def get_clades(Z, labels):
    """提取树中所有的分支集合，用于计算支持度"""
    n = len(labels)
    tree = to_tree(Z)
    clades = []
    
    def traverse(node):
        if node.is_leaf():
            return {labels[node.id]}
        left_leaves = traverse(node.left)
        right_leaves = traverse(node.right)
        current_clade = left_leaves.union(right_leaves)
        clades.append(frozenset(current_clade))
        return current_clade
    
    traverse(tree)
    return clades

def bootstrap_phylo(adata, key, group_col, method='complete', metric='correlation', n_bootstrap=100, sample_number = 100):
    data = bulk(adata, key, group_col)
    labels = data.index.tolist()
    
    # 2. 构建原始树
    Z_orig = build_tree_from_data(data, method, metric)
    orig_clades = get_clades(Z_orig, labels)
    clade_counts = {clade: 0 for clade in orig_clades}
    
    # 3. Bootstrap 循环
    print(f"Starting {n_bootstrap} bootstrap iterations...")
    for i in range(n_bootstrap):
        # 对样本进行重抽样
        boot_data = get_pseudobulk(adata, key, group_col, sample_number = sample_number)
        Z_boot = build_tree_from_data(boot_data,method, metric)
        boot_clades = get_clades(Z_boot, labels)
        
        for clade in orig_clades:
            if clade in boot_clades:
                clade_counts[clade] += 1
                
    # 4. 统计支持度百分比
    support_values = [int((clade_counts[c] / n_bootstrap) * 100) for c in orig_clades]
    return Z_orig, labels, support_values


def plot_tree_with_support(Z, labels, support_values):
    import matplotlib.pyplot as plt
    from Bio import Phylo
    from io import StringIO

    n = len(labels)

    # ---- 计算每个节点的"高度"（从根到该节点的累加距离） ----
    # Z[i, 2] 是第 i 次合并的距离，这些距离是递增的
    node_height = {i: 0.0 for i in range(n)}  # 叶子高度 = 根距离 - 总树高（后面统一处理）

    # 最大距离（根的高度）
    total_height = Z[-1, 2]
    # 叶子到根的距离 = total_height，所以叶子高度应为 0（即叶子在树的最远端）
    # 内部节点 i 的高度 = Z[i, 2]（该节点合并时的距离）

    for i in range(n):
        node_height[i] = total_height  # 叶子节点累加全长为 total_height

    for i, row in enumerate(Z):
        node_height[n + i] = total_height - row[2]  # 内部节点高度：越靠近根高度越小

    # ---- 构建带枝长的 Newick ----
    # 叶子的 Newick: label:branch_length
    # 内部节点的 Newick: (left,right)support:branch_length
    nodes_nwk = {}
    for i in range(n):
        # 叶子到其父节点的枝长
        bl = total_height  # 简化：叶子到根 = total_height，实际父边枝长在父节点合并时计算
        nodes_nwk[i] = f"'{labels[i]}'"

    for i, row in enumerate(Z):
        idx = n + i
        left_idx, right_idx = int(row[0]), int(row[1])
        dist = row[2]
        support = support_values[i]

        # 子节点到当前合并点的枝长
        left_bl = dist - (Z[int(left_idx - n), 2] if left_idx >= n else 0)
        right_bl = dist - (Z[int(right_idx - n), 2] if right_idx >= n else 0)

        # 构造 Newick: (left:bl_left, right:bl_right)Support
        nodes_nwk[idx] = f"({nodes_nwk[left_idx]}:{left_bl:.6f},{nodes_nwk[right_idx]}:{right_bl:.6f}){support}"

    # 根节点的枝长通常省略（或设为 0），这里不加根枝长
    newick_str = nodes_nwk[n + len(Z) - 1] + ";"
    tree = Phylo.read(StringIO(newick_str), "newick")

    # ---- 绘图 ----
    fig = plt.figure(figsize=(12, 8), dpi=100)
    ax = fig.add_subplot(1, 1, 1)

    def get_label(clade):
        if clade.is_terminal():
            return str(clade.name)
        if clade.confidence is not None:
            return f" {int(clade.confidence)}"
        return ""

    Phylo.draw(tree, axes=ax, do_show=False, show_confidence=False,
               label_func=get_label, branch_labels=None)

    ax.set_title("Hierarchical Clustering with Bootstrap Support (%)", pad=20)
    ax.set_ylabel("")
    ax.set_yticks([])
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
Z, labels, supports = bootstrap_phylo(scdata, key= 'X_pca_harmony', 
                                      group_col='sly_anno', 
                                      n_bootstrap=1000, sample_number = 5000, method='complete', metric='correlation')
plot_tree_with_support(Z, labels, supports)

In [ ]:
def print_tree_info(Z, labels, support_values):
    """
    完整输出层次聚类树的结构信息：
    - 每次合并的左右子簇、枝长（距离）、Bootstrap 支持度
    - Newick 字符串（带真实枝长）
    """
    n = len(labels)

    node_names = {i: labels[i] for i in range(n)}
    nodes = {i: f"'{labels[i]}'" for i in range(n)}

    # ---- 逐行输出合并过程 ----
    print("=" * 80)
    print("层次聚类合并过程 (Complete Linkage, Correlation Distance)")
    print("=" * 80)
    print(f"{'Step':>5s}  {'左子簇':>35s}  {'右子簇':>35s}  {'合并距离':>10s}  {'Bootstrap':>10s}")
    print("-" * 80)

    merge_records = []

    for i, row in enumerate(Z):
        left_idx = int(row[0])
        right_idx = int(row[1])
        current_height = row[2]
        support = support_values[i]

        left_name = node_names[left_idx]
        right_name = node_names[right_idx]

        # ---- 计算子节点到当前合并点的边长 ----
        left_height = 0 if left_idx < n else Z[left_idx - n, 2]
        right_height = 0 if right_idx < n else Z[right_idx - n, 2]
        left_edge = current_height - left_height
        right_edge = current_height - right_height

        # ---- 构建带枝长的 Newick 子树 ----
        left_nwk = f"{nodes[left_idx]}:{left_edge:.6f}"
        right_nwk = f"{nodes[right_idx]}:{right_edge:.6f}"
        nodes[n + i] = f"({left_nwk},{right_nwk}){support}"

        # ---- 更新节点名称映射 ----
        node_names[n + i] = f"Node{i}"

        # ---- 输出 ----
        left_display = left_name if left_idx < n else f"[{left_name}]"
        right_display = right_name if right_idx < n else f"[{right_name}]"

        print(f"{i+1:>5d}  {left_display:>35s}  {right_display:>35s}  {current_height:>10.4f}  {support:>7d}%")

        merge_records.append({
            "step": i + 1,
            "left": left_display,
            "right": right_display,
            "distance": current_height,
            "left_edge": left_edge,
            "right_edge": right_edge,
            "support": support,
        })

    print("-" * 80)

    # ---- 按支持度降序排列 ----
    print("\n" + "=" * 80)
    print("按 Bootstrap 支持度降序排列")
    print("=" * 80)
    print(f"{'排名':>4s}  {'合并涉及':>72s}  {'合并距离':>8s}  {'支持度':>8s}")
    print("-" * 80)

    sorted_records = sorted(merge_records, key=lambda x: x["support"], reverse=True)
    for rank, rec in enumerate(sorted_records, 1):
        combination = f"{rec['left']} + {rec['right']}"
        print(f"{rank:>4d}  {combination:<72s}  {rec['distance']:>8.4f}  {rec['support']:>5d}%")

    print("-" * 80)

    # ---- 带枝长的 Newick 字符串 ----
    newick_str = nodes[n + len(Z) - 1] + ";"
    print("\n" + "=" * 80)
    print("Newick 格式字符串（带枝长，可直接用于 FigTree / iTOL / ggtree）")
    print("=" * 80)
    print(newick_str)

    # ---- 枝长说明 ----
    print("\n" + "=" * 80)
    print("枝长详解（每步合并时各子边的实际长度）")
    print("=" * 80)
    print(f"{'Step':>5s}  {'左子边':>35s}  {'右子边':>35s}  {'左边长':>8s}  {'右边长':>8s}  {'合并距离':>8s}")
    print("-" * 80)
    for rec in merge_records:
        print(f"{rec['step']:>5d}  {rec['left']:>35s}  {rec['right']:>35s}  "
              f"{rec['left_edge']:>8.4f}  {rec['right_edge']:>8.4f}  {rec['distance']:>8.4f}")
    print("-" * 80)
    print("说明：叶子到根的总距离 = 各段边长之和，所有叶子总距离相等 = {:.4f}".format(Z[-1, 2]))

    return newick_str


# ===== 调用 =====
newick = print_tree_info(Z, labels, supports)